# Session 3 Lab — Track, Compare & Select an SMS Classifier
**MLOps Essentials · Student lab · Google Colab / Jupyter**

**Mission:** Train two SMS classifiers, log their experiments, compare **validation** precision and recall, then save one traceable model for Session 4.

**What you will do:** Four small coding exercises, one model-selection decision, and an optional advanced experiment. You do **not** need to write the ML algorithm.

**Ground rules**
- Dataset: [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection), with `text` and `label` (`spam`/`ham`).
- Train / validation / test splits stay the same for both runs. **Never use test results to choose a model.**
- The model is the **complete fitted** `TfidfVectorizer + MultinomialNB` pipeline.
- Use **synthetic** custom SMS messages; do not upload private conversations or load unknown model files.
- If MLflow cannot install, use the notebook's local comparison-table fallback.

**How to work:** Read each step in this order: **WHY → WHAT → RUN → VERIFY → EXPLAIN**. Predict before you run.

## Step 0 — Prepare the environment
**WHY:** We need tools for training, measuring and recording experiments.

**WHAT:** Check the required packages; try to install MLflow if it is missing. MLflow is optional for completing the core lab, but important to explore when available.

**Before running:** Which tool builds our model, and which tool records experiments?

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

if importlib.util.find_spec("mlflow") is None and os.environ.get("SESSION3_SKIP_MLFLOW_INSTALL") != "1":
    print("Installing MLflow...")
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow==3.5.1"],
                       check=True, timeout=180)
    except (Exception) as err:
        print("MLflow install unavailable. We will use the comparison-table fallback.")
        print(type(err).__name__, str(err)[:160])

import pandas as pd
import joblib, sklearn
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import precision_score, recall_score

try:
    import mlflow
    import mlflow.sklearn
    # MLflow 3.x uses log_model(name=...) and returns a model URI.
    MLFLOW_READY = int(mlflow.__version__.split(".")[0]) == 3
    if not MLFLOW_READY:
        print("MLflow 3.x required for logging in this notebook; local fallback enabled.")
except Exception as err:
    MLFLOW_READY = False
    print("MLflow unavailable:", type(err).__name__)

OUT = Path("session3_outputs")
OUT.mkdir(exist_ok=True)
if MLFLOW_READY:
    mlflow.set_tracking_uri((OUT / "mlruns").resolve().as_uri())
    mlflow.set_experiment("session3_sms_spam")

print("scikit-learn:", sklearn.__version__)
print("MLflow tracking:", "available" if MLFLOW_READY else "fallback: CSV + joblib")

**VERIFY:** Package versions appear and you see either MLflow tracking or fallback status.

**EXPLAIN:** scikit-learn trains and evaluates; MLflow tracks run metadata and saved models. Missing MLflow does not prevent completing the lab.

**If installation fails:** Continue. Your experiment results will still be saved as CSV and the model as a `joblib` file.

### Helper — tables that explain themselves
**WHY:** From here on, some steps show a **live table** built from *your own* run, followed by a **glossary** that says what each term means and what a *higher* or *lower* value tells you.

**RUN:** Run this cell once. It only defines helpers: nothing is trained and nothing is printed. You do not need to memorize it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown
from sklearn.metrics import confusion_matrix

# Colour-blind-friendly colours used by every chart in this notebook.
BLUE, ORANGE, GREY = "#0072B2", "#E69F00", "#777777"

def show_table(df, formats=None):
    """Show a live table without the row-number column."""
    display(df.style.hide(axis="index").format(formats or None))

def show_glossary(rows):
    """Show a 'Term | Meaning | Higher means | Lower means' table below a live table."""
    glossary = pd.DataFrame(rows, columns=["Term", "Meaning", "Higher means", "Lower means"])
    styled = (glossary.style.hide(axis="index")
              .set_properties(**{"text-align": "left", "white-space": "normal"})
              .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
    display(styled)

def error_counts(fitted_pipeline, texts, labels):
    """Count the two mistake types (and total spam alerts) for a fitted pipeline."""
    predicted = fitted_pipeline.predict(texts)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=["ham", "spam"]).ravel()
    return {"false_alarms": int(fp), "missed_spam": int(fn),
            "caught_spam": int(tp), "spam_alerts": int(fp + tp)}

## Step 1 — Inspect the SMS dataset · Exercise 1
**WHY:** A model can look accurate while missing most spam when one class is more common.

**WHAT:** Load the official tab-separated SMS collection, normalize its two columns to `label` and `text`, and remove duplicate message texts **before** splitting.

**Predict:** Will `spam` or `ham` be more common?

**RUN:** The first cell downloads and checks the data. The second is your small coding exercise.

In [ ]:
import csv, io, zipfile, urllib.request

SOURCE_URL = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
source_file = Path("SMSSpamCollection")

if not source_file.exists():
    try:
        with urllib.request.urlopen(SOURCE_URL, timeout=35) as reply:
            archive_data = reply.read()
        with zipfile.ZipFile(io.BytesIO(archive_data)) as archive:
            match = next((name for name in archive.namelist()
                          if name.split("/")[-1] == "SMSSpamCollection"), None)
            if match is None:
                raise ValueError("The UCI archive has no 'SMSSpamCollection' file")
            source_file.write_bytes(archive.read(match))
    except Exception as err:
        raise RuntimeError(
            "Download failed. From the UCI dataset page, download the ZIP, extract "
            "'SMSSpamCollection', upload that file next to the notebook, "
            "and rerun this cell."
        ) from err

# Original file: no header; first column is a label, second is SMS text.
try:
    raw = pd.read_csv(source_file, sep="\t", header=None, names=["label", "text"],
                      quoting=csv.QUOTE_NONE, encoding="utf-8")
except UnicodeDecodeError:
    raw = pd.read_csv(source_file, sep="\t", header=None, names=["label", "text"],
                      quoting=csv.QUOTE_NONE, encoding="latin-1")
raw["label"] = raw["label"].astype(str).str.strip().str.lower()
raw = raw.dropna(subset=["text"]).copy()
raw["text"] = raw["text"].astype(str).str.strip()
assert set(raw["label"]) == {"ham", "spam"}, "Unexpected source format/labels"

# Remove ambiguous duplicate texts with conflicting labels; keep one copy per other text.
conflicts = raw.groupby("text")["label"].nunique()
ambiguous_texts = set(conflicts[conflicts > 1].index)
df = raw.loc[~raw["text"].isin(ambiguous_texts)].drop_duplicates("text").reset_index(drop=True)
assert df["text"].is_unique and df["label"].notna().all()
print("Source rows:", len(raw), "| Unique usable messages:", len(df))
print("Removed conflicting texts:", len(ambiguous_texts))
print("Columns:", list(df.columns))
print("Counts:", df["label"].value_counts().to_dict())

In [ ]:
# EXERCISE 1 — Count a class and calculate its share.
label_to_inspect = "spam"  # ✏️ Change to "ham" and rerun. What changes?
class_count = int((df["label"] == label_to_inspect).sum())
class_percent = 100 * class_count / len(df)
print(f"{label_to_inspect}: {class_count} messages ({class_percent:.1f}% of usable data)")
assert 0 <= class_percent <= 100

**VERIFY:** You see two classes and many more ham messages than spam; after cleaning, the row count may be less than the UCI catalog's 5,574.

**EXPLAIN:** Why is **spam recall** important when spam messages are less common?

**Hint:** Class percentage = `(number in class / total messages) × 100`. Change only `label_to_inspect` for this exercise.

### See it — class balance (live table + chart)
**WHY:** Counts hide the imbalance; a labelled table and a bar chart make it obvious.

**RUN:** Build both from your own `df`. **VERIFY:** The bars match the counts printed above. **EXPLAIN:** Use the glossary to say why *accuracy alone* is a poor score here.

In [ ]:
balance = df["label"].value_counts().rename_axis("Class").reset_index(name="Messages")
balance["Share of data"] = balance["Messages"] / balance["Messages"].sum()
show_table(balance, {"Messages": "{:,}", "Share of data": "{:.1%}"})

fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.bar(balance["Class"], balance["Messages"],
              color=[BLUE if name == "ham" else ORANGE for name in balance["Class"]])
ax.bar_label(bars, labels=[f"{n:,}  ({s:.1%})" for n, s in zip(balance["Messages"], balance["Share of data"])])
ax.set_ylim(0, balance["Messages"].max() * 1.15)
ax.set_ylabel("Messages")
ax.set_title("Ham vs spam in the cleaned data")
plt.show()

ham_share = float(balance.loc[balance["Class"] == "ham", "Share of data"].iloc[0])
print(f"An always-ham filter would be {ham_share:.1%} accurate and catch 0% of spam.\n")

show_glossary([
    ("Messages", "Number of cleaned messages in the class.",
     "More examples to learn from, and more to measure on.",
     "Fewer examples: patterns are harder to learn and scores are noisier."),
    ("Share of data", "Class messages ÷ all messages × 100.",
     "A common class. Always guessing it scores about this share as accuracy.",
     "A rare class. A model can ignore it and still look accurate."),
    ("Class imbalance", "How far apart the class shares are.",
     "Accuracy becomes misleading, so check spam precision and spam recall.",
     "Accuracy says more about the model."),
])

## Step 2 — Split once and keep the test set locked
**WHY:** Both runs need the **same** training and validation messages for a fair comparison.

**WHAT:** Use a fixed random seed and stratified splits, keeping approximately **60% train / 20% validation / 20% test**.

**Predict:** Which subset should help us choose between Run A and Run B? Which must stay untouched?

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    df["text"], df["label"], test_size=0.20, stratify=df["label"], random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)
print("Train:", len(X_train), "| Validation:", len(X_val), "| Test: locked")
assert len(X_train) + len(X_val) + len(X_test) == len(df)
assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

**VERIFY:** The three groups add up to the number of cleaned SMS messages. No test metrics appear.

**EXPLAIN:** Training data teaches the model; validation helps choose it; test is reserved for one final assessment **after** selection. TF-IDF must learn its vocabulary from training data only.

### See it — what each subset is for (live table + chart)
**WHAT:** Size and spam share of the three subsets, built from your split. Test is listed only so you can see its **size**: its results stay locked until Step 9.

**VERIFY:** The three sizes add up to the cleaned total, and the spam share is similar in every subset. **EXPLAIN:** Which subset may we use to choose between runs?

In [ ]:
subsets = {"Training": y_train, "Validation": y_val, "Test": y_test}
roles = {
    "Training": ("Teaches the model", "No - it only learns here"),
    "Validation": ("Compares Run A and Run B", "Yes - choose here"),
    "Test": ("One final check (Step 9)", "No - locked until the end"),
}
split_rows = []
for subset_name, subset_labels in subsets.items():
    split_rows.append({
        "Subset": subset_name,
        "Messages": len(subset_labels),
        "Share of all": len(subset_labels) / len(df),
        "Spam messages": int((subset_labels == "spam").sum()),
        "Spam share": float((subset_labels == "spam").mean()),
        "Used for": roles[subset_name][0],
        "Choose a model with it?": roles[subset_name][1],
    })
split_table = pd.DataFrame(split_rows)
show_table(split_table, {"Messages": "{:,}", "Share of all": "{:.1%}", "Spam share": "{:.1%}"})

ham_counts = split_table["Messages"] - split_table["Spam messages"]
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(split_table["Subset"], ham_counts, color=BLUE, label="ham")
ax.barh(split_table["Subset"], split_table["Spam messages"], left=ham_counts, color=ORANGE, label="spam")
for position, (total, share) in enumerate(zip(split_table["Messages"], split_table["Spam share"])):
    ax.text(total + 25, position, f"{total:,} messages, spam {share:.1%}", va="center")
ax.set_xlim(0, split_table["Messages"].max() * 1.6)
ax.invert_yaxis()
ax.set_xlabel("Messages")
ax.set_title("Ham/spam mix in each subset")
ax.legend(loc="lower right")
plt.show()

spread = 100 * (split_table["Spam share"].max() - split_table["Spam share"].min())
print(f"Spam share differs by only {spread:.2f} percentage points across subsets (stratify keeps them alike).")
print(f"Each spam message in validation moves spam recall by about {1 / split_table.loc[1, 'Spam messages']:.3f}.\n")

show_glossary([
    ("Messages", "Number of messages in the subset.",
     "Scores measured on it are steadier.",
     "One message moves the scores more."),
    ("Spam messages", "Real spam inside the subset.",
     "Recall is measured on more cases, so each miss changes it less.",
     "Each missed spam changes recall a lot (about 1 ÷ this number)."),
    ("Spam share", "Spam messages ÷ messages in the subset.",
     "Much higher than the other subsets: spam is over-represented, so scores are not comparable.",
     "Much lower than the other subsets: spam is under-represented, so scores are not comparable."),
])

## Step 3 — Train Run A
**WHY:** We need a baseline result to compare against a second attempt.

**WHAT:** Train a fresh `Pipeline(TfidfVectorizer(), MultinomialNB(alpha=...))`, score it on validation data and log **alpha, precision, recall and the fitted pipeline**.

**Predict:** Will the classifier catch every spam message? Why might some mistakes remain?

In [ ]:
# Shared helper: students use it but do not need to memorize how it works.
RUN_MODELS, RUN_RECORDS = {}, {}

def train_and_record(run_name, alpha, min_df=1, fit_prior=True):
    fitted_pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(min_df=min_df)),
        ("classifier", MultinomialNB(alpha=alpha, fit_prior=fit_prior)),
    ])
    fitted_pipeline.fit(X_train, y_train)  # TF-IDF learns ONLY from training data.
    val_predictions = fitted_pipeline.predict(X_val)
    precision = float(precision_score(y_val, val_predictions, pos_label="spam", zero_division=0))
    recall = float(recall_score(y_val, val_predictions, pos_label="spam", zero_division=0))
    record = {"run": run_name, "alpha": float(alpha), "min_df": int(min_df), "fit_prior": bool(fit_prior),
              "val_spam_precision": precision, "val_spam_recall": recall,
              "mlflow_run_id": "not_logged", "mlflow_model_uri": "not_logged",
              "mlflow_status": "fallback"}
    if MLFLOW_READY:
        try:
            with mlflow.start_run(run_name=run_name) as active_run:
                mlflow.log_param("alpha", alpha)
                mlflow.log_param("min_df", min_df)
                mlflow.log_param("fit_prior", fit_prior)
                mlflow.log_metric("val_spam_precision", precision)
                mlflow.log_metric("val_spam_recall", recall)
                info = mlflow.sklearn.log_model(fitted_pipeline, name="sms_pipeline")
                record["mlflow_run_id"] = active_run.info.run_id
                record["mlflow_model_uri"] = info.model_uri
                record["mlflow_status"] = "logged"
        except Exception as err:
            print("MLflow logging problem; keeping local results:", type(err).__name__, str(err)[:160])
            record["mlflow_status"] = "logging_failed"
    RUN_MODELS[run_name] = fitted_pipeline
    RUN_RECORDS[run_name] = record
    print(f"{run_name}: precision={precision:.3f}, recall={recall:.3f}, MLflow={record['mlflow_status']}")
    return record

ALPHA_A = 0.5
run_a = train_and_record("Run A", alpha=ALPHA_A)

**VERIFY:** Run A prints validation spam precision, validation spam recall, and an MLflow status.

**EXPLAIN:** A **parameter** records what we chose; a **metric** records what we measured; the saved pipeline is a **model artifact**.

**Note:** `alpha` is a model *hyperparameter*, recorded as an MLflow *parameter*. Do not compare to the test set.

### See it — what was recorded for Run A (live table)
**WHAT:** Everything a run logs is a **parameter**, a **metric** or an **artifact**. The first table lists what Run A recorded; the second explains the three kinds.

**VERIFY:** The values match the line printed above. **EXPLAIN:** Which kind answers *what did we choose?*, which *how well did it work?* and which *what can I reload?*

In [ ]:
tfidf_of_run_a = RUN_MODELS["Run A"].named_steps["tfidf"]
recorded = pd.DataFrame([
    ("Parameter", "alpha", str(run_a["alpha"]), "Naive Bayes smoothing we chose"),
    ("Parameter", "min_df", str(run_a["min_df"]), "How rare a word may be and still be kept (1 = keep every word)"),
    ("Metric", "val_spam_precision", f"{run_a['val_spam_precision']:.3f}", "Share of spam alerts that were real spam (validation)"),
    ("Metric", "val_spam_recall", f"{run_a['val_spam_recall']:.3f}", "Share of real spam that was caught (validation)"),
    ("Artifact", "sms_pipeline", f"TF-IDF ({len(tfidf_of_run_a.vocabulary_):,} words) + Naive Bayes",
     "The fitted model that can be reloaded"),
], columns=["Kind", "Name", "Value in your Run A", "What it records"])
show_table(recorded)

display(Markdown("**What the three kinds mean**"))
show_table(pd.DataFrame([
    ("Parameter", "A setting you chose before training", "alpha", "What did we choose?"),
    ("Metric", "A result you measured after training", "val_spam_recall", "How well did it work?"),
    ("Artifact", "A file the run produced", "sms_pipeline", "What can I reload?"),
], columns=["Kind", "Meaning", "Example in this lab", "Question it answers"]))

## Step 4 — Train Run B · Exercise 2
**WHY:** We can learn whether changing one setting changes validation performance.

**WHAT:** Train with the **same data and algorithm**, changing only `alpha`.

**Predict:** Will increasing `alpha` necessarily improve precision or recall?

In [ ]:
# EXERCISE 2 — Configure the second run.
ALPHA_B = 1.0  # ✏️ Confirm this is different from Run A's 0.5.
assert ALPHA_B != ALPHA_A, "Run B needs a different alpha value"
run_b = train_and_record("Run B", alpha=ALPHA_B)

results_df = pd.DataFrame([RUN_RECORDS["Run A"], RUN_RECORDS["Run B"]])
results_df.to_csv(OUT / "run_results.csv", index=False)
display(results_df[["run", "alpha", "val_spam_precision", "val_spam_recall", "mlflow_status"]])

**VERIFY:** The table shows exactly **two core runs**, with `alpha=0.5` and `alpha=1.0`.

**EXPLAIN:** Did the scores change? If not, that is a valid finding: different settings can still produce the same predicted labels.

**Hint:** Change just the `ALPHA_B` setting. Do not change the data split or use `X_test` here.

### See it — compare Run A and Run B (live table + chart)
**WHAT:** The table above shows two scores. This one adds the **mistake counts** behind them, so you can see *how many real messages* each run got wrong on validation.

**VERIFY:** Precision and recall match the table above. **EXPLAIN:** Use the glossary to say, in one sentence, what changing `alpha` did.

In [ ]:
comparison_rows = []
for run_name in ["Run A", "Run B"]:
    run_record = RUN_RECORDS[run_name]
    counts = error_counts(RUN_MODELS[run_name], X_val, y_val)
    comparison_rows.append({
        "Run": run_name,
        "alpha": run_record["alpha"],
        "Spam precision": run_record["val_spam_precision"],
        "Spam recall": run_record["val_spam_recall"],
        "False alarms": counts["false_alarms"],
        "Missed spam": counts["missed_spam"],
        "Spam alerts": counts["spam_alerts"],
    })
mistakes_table = pd.DataFrame(comparison_rows)
show_table(mistakes_table, {"alpha": "{:g}", "Spam precision": "{:.3f}", "Spam recall": "{:.3f}"})

positions = np.arange(len(mistakes_table))
width = 0.38
run_labels = [f"{r} (alpha={a})" for r, a in zip(mistakes_table["Run"], mistakes_table["alpha"])]

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
precision_bars = left.bar(positions - width / 2, mistakes_table["Spam precision"], width,
                          color=BLUE, label="Spam precision")
recall_bars = left.bar(positions + width / 2, mistakes_table["Spam recall"], width,
                       color=ORANGE, label="Spam recall")
left.bar_label(precision_bars, fmt="%.3f")
left.bar_label(recall_bars, fmt="%.3f")
left.set_xticks(positions, run_labels)
left.set_ylim(0, 1.15)
left.set_title("Validation scores (higher is better)")
left.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)

false_bars = right.bar(positions - width / 2, mistakes_table["False alarms"], width,
                       color=BLUE, label="False alarms (real SMS blocked)")
missed_bars = right.bar(positions + width / 2, mistakes_table["Missed spam"], width,
                        color=ORANGE, label="Missed spam (spam delivered)")
right.bar_label(false_bars)
right.bar_label(missed_bars)
right.set_xticks(positions, run_labels)
right.set_ylim(0, max(1, mistakes_table[["False alarms", "Missed spam"]].to_numpy().max()) * 1.2)
right.set_title("Mistakes on validation (lower is better)")
right.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
fig.tight_layout()
plt.show()
print("Colours pair up: blue = precision <-> false alarms; orange = recall <-> missed spam.\n")

show_glossary([
    ("alpha", "Smoothing added to every word count while the model learns. It is the only setting that differs between the runs.",
     "Rare and unseen words matter less. The model tends to become more cautious and flag less spam.",
     "Small word counts are trusted more. The model tends to flag more spam but leans on rare words."),
    ("Spam precision", "Of the messages flagged as spam, the share that really are spam.",
     "Fewer legitimate messages are blocked.",
     "More legitimate messages are blocked."),
    ("Spam recall", "Of all real spam, the share the model flagged.",
     "Less spam reaches the user.",
     "More spam reaches the user."),
    ("False alarms", "Legitimate messages wrongly flagged as spam (false positives).",
     "Worse: more real messages are blocked.",
     "Better: fewer real messages are blocked."),
    ("Missed spam", "Real spam that was not flagged (false negatives).",
     "Worse: more spam is delivered.",
     "Better: less spam is delivered."),
    ("Spam alerts", "All messages flagged as spam (false alarms + caught spam).",
     "More blocking. Good only if the extra alerts are real spam.",
     "Less blocking. Safer for real messages, but more spam gets through."),
])

### Interpreting the Run A vs Run B comparison

*Reference results (seed 42, `alpha` 0.5 and 1.0, 1,032 validation messages of which 128 are spam). If you kept the defaults you should see the same numbers.*

| Measure | Run A (`alpha` 0.5) | Run B (`alpha` 1.0) |
|---|---:|---:|
| Spam precision | 1.000 | 1.000 |
| Spam recall | 0.750 | 0.633 |
| False alarms | 0 | 0 |
| Missed spam | 32 | 47 |
| Spam alerts | 96 | 81 |

**What this means:** Neither run blocked a legitimate validation message, so precision is tied at 1.0 and cannot separate them. The difference is recall: Run A caught 96 of the 128 spam messages and Run B caught 81, so raising `alpha` let **15 more spam messages through**. A tied precision is not a tie overall; always read precision and recall together. With only 128 spam messages in validation, each one moves recall by about 0.008.

## Step 5 — Inspect your experiment records · Exercise 3
**WHY:** We should be able to identify which settings and metrics belong to which model **later**, not just while the notebook is open.

**WHAT:** Inspect a chosen run using MLflow when available, or inspect the exported comparison table.

**Predict:** What information should be attached to each run ID?

In [ ]:
# EXERCISE 3 — Change the selected name to "Run B" and inspect its record too.
inspect_run = "Run A"  # ✏️ Change this to "Run B" and rerun.
record = RUN_RECORDS[inspect_run]
print("Selected record:", inspect_run)
print("MLflow run ID:", record["mlflow_run_id"])
print("Logged model URI:", record["mlflow_model_uri"])
if MLFLOW_READY and record["mlflow_status"] == "logged":
    tracked = mlflow.get_run(record["mlflow_run_id"])
    print("MLflow parameters:", tracked.data.params)
    print("MLflow metrics:", tracked.data.metrics)
else:
    print("Local record (MLflow fallback):")
    display(pd.DataFrame([record])[["run", "alpha", "min_df", "val_spam_precision", "val_spam_recall"]])

**VERIFY:** Run A and Run B have their own records. When MLflow succeeds, it also provides unique run IDs and logged model URIs; otherwise, the CSV keeps the comparison traceable locally.

**EXPLAIN:** Why would looking at only the **latest metric** without a run name and parameter be misleading?

**Hint:** `inspect_run` must exactly match an existing name.

### See it — the full record behind each run (live table)
**WHY:** A score without its run name and settings cannot be traced later.

**VERIFY:** One column per run, same fields. **EXPLAIN:** If two runs reported the same recall, which rows would you check first?

In [ ]:
record_fields = [
    ("Run name", "run", "Which attempt is this?"),
    ("alpha", "alpha", "Which smoothing setting was used?"),
    ("min_df", "min_df", "Which words were kept?"),
    ("Validation spam precision", "val_spam_precision", "How trustworthy were the spam alerts?"),
    ("Validation spam recall", "val_spam_recall", "How much spam was caught?"),
    ("MLflow status", "mlflow_status", "Was the run logged, or kept only locally?"),
    ("MLflow run ID (first 8 characters)", "mlflow_run_id", "The unique key for finding this run again"),
    ("Model URI", "mlflow_model_uri", "Where the saved pipeline can be reloaded from"),
]

def show_value(key, value):
    if key.startswith("val_"):
        return f"{value:.3f}"
    if key == "mlflow_run_id" and value != "not_logged":
        return value[:8] + "..."
    return str(value)

full_record = pd.DataFrame({
    "Field": [label for label, _, _ in record_fields],
    "Run A": [show_value(key, RUN_RECORDS["Run A"][key]) for _, key, _ in record_fields],
    "Run B": [show_value(key, RUN_RECORDS["Run B"][key]) for _, key, _ in record_fields],
    "Question it answers later": [question for _, _, question in record_fields],
})
show_table(full_record)

**Why a bare number is not enough**

| What you remember | What you cannot answer later |
|---|---|
| "Recall was 0.633" | Which run? Which `alpha`? Which data split? |
| "It was Run B" | What did it score? Where is the saved file? |
| Run ID + parameters + metrics + model URI | Nothing is missing: reload it, compare it, and defend the choice |

## Step 6 — Choose a candidate
**WHY:** Better depends on what matters to the application. An SMS filter that silently blocks legitimate messages may favor **precision**.

**WHAT:** Compare the validation scores, choose Run A or Run B, and write your reason. Do not inspect test data.

**Predict:** Which model would you choose for automatic blocking? Would a review folder change your preference?

In [ ]:
comparison = results_df[["run", "alpha", "val_spam_precision", "val_spam_recall"]]
display(comparison)

# ✏️ Choose based on the real validation results above, not on a hypothetical example.
# Change "AUTO" to "Run A" or "Run B" after comparing the table.
candidate_name = "AUTO"
selection_reason = "Prefer higher validation spam precision; break ties using recall."

recommended = results_df.sort_values(
    ["val_spam_precision", "val_spam_recall"],
    ascending=[False, False], kind="stable"
).iloc[0]["run"]
if candidate_name == "AUTO":
    candidate_name = recommended
    print("Automatic classroom recommendation used. Explain whether you agree.")
assert candidate_name in ("Run A", "Run B")
assert selection_reason.strip(), "Write a reason for your choice"
selected = RUN_RECORDS[candidate_name]
selected_pipeline = RUN_MODELS[candidate_name]
print("CANDIDATE:", candidate_name, "| alpha:", selected["alpha"])
print("REASON:", selection_reason)

**VERIFY:** You have exactly one selected candidate and an explanation based on validation metrics.

**EXPLAIN:** Why is this a **candidate**, not automatically a production-ready model?

**Hint:** A useful rule for automatic blocking is precision first, then recall if precision ties. Another application may use a different rule.

### See it — one set of results, two different goals (live table)
**WHY:** "Better" depends on what a mistake costs. The same two runs are ranked here by two rules.

**VERIFY:** The *Automatic blocking* winner equals the AUTO recommendation above. **EXPLAIN:** Would a review folder change your pick? Why, or why not?

In [ ]:
ranking_rules = [
    ("Automatic blocking", "Blocking a real message is the costly mistake",
     "val_spam_precision", "val_spam_recall", "precision, then recall"),
    ("Review folder", "Flagged SMS go to a folder, so a false alarm is cheap; missed spam is the costly mistake",
     "val_spam_recall", "val_spam_precision", "recall, then precision"),
]
scenario_rows = []
for scenario, matters, first_metric, second_metric, rule_text in ranking_rules:
    ranked = results_df.sort_values([first_metric, second_metric], ascending=[False, False], kind="stable")
    top = ranked.iloc[0]
    scenario_rows.append({
        "Scenario": scenario,
        "What matters most": matters,
        "Ranking rule": rule_text,
        "First metric tied?": "Yes - second metric decides" if ranked[first_metric].nunique() == 1 else "No",
        "Winner": top["run"],
        "Spam precision": top["val_spam_precision"],
        "Spam recall": top["val_spam_recall"],
    })
scenario_table = pd.DataFrame(scenario_rows)
show_table(scenario_table, {"Spam precision": "{:.3f}", "Spam recall": "{:.3f}"})
print("Automatic-blocking winner matches the AUTO recommendation above:",
      scenario_rows[0]["Winner"] == recommended)

### Interpreting the two goals

*Reference results (seed 42, Run A `alpha` 0.5 vs Run B `alpha` 1.0).*

| Scenario | Ranking rule | Winner | Why |
|---|---|---|---|
| Automatic blocking | precision, then recall | Run A | Precision is tied at 1.000, so recall decides: 0.750 vs 0.633. |
| Review folder | recall, then precision | Run A | Recall is higher (0.750 vs 0.633), and precision is tied. |

**What this means:** On these data both goals pick the same run, because Run A is at least as good on both measures. That will not always happen. If one run had higher precision and the other higher recall, the application would have to decide which mistake it can afford. The winner is still only a **candidate**: it has not been checked on the test set. **Bonus Step 10** creates a run that makes the two goals disagree.

## Step 7 — Save, reload and test custom SMS · Exercise 4
**WHY:** A selected model is useful only if another program can load it and classify raw SMS text.

**WHAT:** Save the **whole fitted pipeline**, reload it and try synthetic messages that you write yourself.

**Predict:** Which of your messages will be labeled spam? Could a surprising result still be plausible?

In [ ]:
import json, platform

model_path = OUT / "selected_pipeline.joblib"
joblib.dump(selected_pipeline, model_path)
# SECURITY: Only load a joblib file YOU created or from a source you trust.
reloaded_pipeline = joblib.load(model_path)

# EXERCISE 4 — Change at least one synthetic sentence, then rerun.
custom_sms = [
    "Hi! Can we meet at the library tomorrow?",
    "Congratulations! Claim your free prize now!",
    "Please send the meeting notes when you can.",  # ✏️ Edit this synthetic SMS.
]
custom_predictions = reloaded_pipeline.predict(custom_sms)
for sms, prediction in zip(custom_sms, custom_predictions):
    print(f"{prediction.upper():4}  |  {sms}")

assert hasattr(reloaded_pipeline.named_steps["tfidf"], "vocabulary_")
assert hasattr(reloaded_pipeline.named_steps["classifier"], "classes_")
assert len(custom_predictions) == len(custom_sms)

selection_note = {
    "project": "sms_spam_classifier",
    "candidate_id": "session3_candidate_v1",  # Local handoff label, NOT a registry version.
    "selected_run": candidate_name,
    "alpha": selected["alpha"],
    "min_df": selected["min_df"],
    "validation_spam_precision": selected["val_spam_precision"],
    "validation_spam_recall": selected["val_spam_recall"],
    "selection_reason": selection_reason,
    "mlflow_run_id": selected["mlflow_run_id"],
    "mlflow_model_uri": selected["mlflow_model_uri"],
    "artifact_file": model_path.name,
    "random_seed": 42,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
    "test_used_for_selection": False,
}
(OUT / "selection_note.json").write_text(json.dumps(selection_note, indent=2), encoding="utf-8")
print("Saved:", model_path, "and selection_note.json")

**VERIFY:** Predictions appear for your synthetic SMS; both fitted TF-IDF and classifier are present; a model file and selection note exist.

**EXPLAIN:** Why would saving only `MultinomialNB` be insufficient to handle a raw SMS string?

**Hint:** The classifier needs the numerical features produced by the **fitted** TF-IDF vectorizer. Do not expect every synthetic phrase to be predicted correctly.

### See it — how strongly did the model lean? (live table + chart)
**WHY:** `predict()` prints only a label. The **spam score** shows how strongly the model leaned.

**RUN:** Uses the reloaded pipeline and your `custom_sms`. **EXPLAIN:** Which message came closest to the 0.50 line? Is it the one you expected to be ambiguous?

In [ ]:
from matplotlib.patches import Patch

custom_classes = list(reloaded_pipeline.classes_)
custom_spam_col = custom_classes.index("spam")
custom_tfidf = reloaded_pipeline.named_steps["tfidf"]
custom_analyzer = custom_tfidf.build_analyzer()
custom_scores = reloaded_pipeline.predict_proba(custom_sms)[:, custom_spam_col]

def known_words(message):
    words = custom_analyzer(message)
    known = sum(word in custom_tfidf.vocabulary_ for word in words)
    return f"{known} of {len(words)}"

custom_table = pd.DataFrame({
    "SMS": custom_sms,
    "Spam score": custom_scores,
    "Decision (score >= 0.50)": ["SPAM" if score >= 0.5 else "HAM" for score in custom_scores],
    "Known words used": [known_words(message) for message in custom_sms],
})
show_table(custom_table, {"Spam score": "{:.3f}"})

short_labels = [m if len(m) <= 45 else m[:42] + "..." for m in custom_sms]
fig, ax = plt.subplots(figsize=(8.5, 1.6 + 0.55 * len(custom_sms)))
score_bars = ax.barh(short_labels, custom_scores,
                     color=[ORANGE if score >= 0.5 else BLUE for score in custom_scores])
ax.axvline(0.5, color=GREY, linestyle="--")
ax.bar_label(score_bars, fmt="%.3f", padding=3)
ax.set_xlim(0, 1.12)
ax.invert_yaxis()
ax.set_xlabel("Spam score (0 = ham-like, 1 = spam-like); dashed line = 0.50 cut-off")
ax.set_title("Spam score of each custom SMS")
ax.legend(handles=[Patch(color=ORANGE, label="flagged as spam"), Patch(color=BLUE, label="kept as ham")],
          loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2)
fig.tight_layout()
plt.show()
print()

show_glossary([
    ("Spam score", "The model's spam estimate for the message (`predict_proba`). Not automatically a calibrated probability.",
     "The words resemble training spam. Near 1.0 = very spam-like.",
     "The words resemble training ham. Near 0.0 = very ham-like."),
    ("Cut-off (0.50)", "The rule that turns a score into a label.",
     "Only strongly spam-like messages are flagged: fewer false alarms, more missed spam.",
     "More messages are flagged: more false alarms, less missed spam."),
    ("Known words used", "How many of the message's words exist in the training vocabulary.",
     "More evidence for the model, so the score is better supported.",
     "Unknown words are ignored, so the score rests on few words and is shakier."),
])

### See it — why the *whole* pipeline must be saved (live)
**WHY:** The classifier cannot read text. It only understands the numbers made by the **fitted** TF-IDF stage.

**YOUR TURN:** Change `explain_index` to take a different custom SMS apart. **EXPLAIN:** What would go wrong if you saved only `MultinomialNB`?

In [ ]:
explain_index = 1  # ✏️ YOUR TURN: 0, 1 or 2 - which custom SMS to take apart
assert 0 <= explain_index < len(custom_sms), "Pick a number that matches a message in custom_sms"
message = custom_sms[explain_index]

tfidf_stage = reloaded_pipeline.named_steps["tfidf"]
bayes_stage = reloaded_pipeline.named_steps["classifier"]
stage_classes = [str(name) for name in reloaded_pipeline.classes_]
spam_col, ham_col = stage_classes.index("spam"), stage_classes.index("ham")

# 1) The two stages that were saved together.
show_table(pd.DataFrame([
    ("1. TfidfVectorizer", "Raw SMS text",
     f"A vector of {len(tfidf_stage.vocabulary_):,} numbers (one per vocabulary word)",
     "The vocabulary and the word weights"),
    ("2. MultinomialNB", "That numeric vector",
     f"A label from {stage_classes} and a spam score",
     "Which words lean toward spam or ham"),
], columns=["Saved stage", "Takes in", "Gives out", "Learned from training data"]))

# 2) Take one message apart.
words = tfidf_stage.build_analyzer()(message)
known = sorted({w for w in words if w in tfidf_stage.vocabulary_})
ignored = sorted({w for w in words if w not in tfidf_stage.vocabulary_})
vector = tfidf_stage.transform([message])
spam_lean = bayes_stage.feature_log_prob_[spam_col] - bayes_stage.feature_log_prob_[ham_col]

print("Message:", message)
print(f"Spam score: {reloaded_pipeline.predict_proba([message])[0, spam_col]:.3f}")
print("Words ignored (never seen in training):", ignored if ignored else "none")
if known:
    word_table = pd.DataFrame({
        "Word": known,
        "TF-IDF weight": [vector[0, tfidf_stage.vocabulary_[w]] for w in known],
        "Spam lean": [spam_lean[tfidf_stage.vocabulary_[w]] for w in known],
    })
    word_table["Pushes toward"] = np.where(word_table["Spam lean"] > 0, "spam", "ham")
    show_table(word_table.sort_values("Spam lean", ascending=False),
               {"TF-IDF weight": "{:.3f}", "Spam lean": "{:+.2f}"})
print()

show_glossary([
    ("TF-IDF weight", "How important a word is in this message: frequent here, but rare across training messages.",
     "A distinctive word that gets a bigger say in the score.",
     "A common word (or one of many words) that gets a smaller say."),
    ("Spam lean", "log( P(word | spam) / P(word | ham) ), learned by Naive Bayes from the training messages.",
     "Positive and large: the word is typical of spam.",
     "Negative: typical of ham. Near 0: the word tells the model little."),
    ("Words ignored", "Words in the message that the TF-IDF vocabulary never saw during training.",
     "Less evidence: the prediction rests on fewer words.",
     "The model has seen every word, so the message is well covered."),
])

## Step 8 — OPTIONAL ADVANCED CHALLENGE: Remove rare words
**WHY:** Simplifying the learned vocabulary may reduce complexity, but it might also remove useful spam clues.

**WHAT:** Keep the selected `alpha` and splits fixed. Compare the original pipeline (`min_df=1`) with a new pipeline (`min_df=3`). `min_df` is the minimum number of training documents containing a term.

**Predict:** Will the vocabulary be smaller? Will precision or recall necessarily improve?

**Your challenge:** Modify the one marked setting, run a third experimental model, compare vocabulary sizes and validation scores, and explain the trade-off. **Do not use the test set and do not replace the core selected candidate.**

### See it — which words does `min_df` remove? (live table)
**WHAT:** `min_df=3` keeps a word only if it appears in **at least 3 training messages**. The table counts real messages from your training set and shows what each setting does to a few words.

**YOUR TURN:** Add your own words to `words_to_check` and rerun. **EXPLAIN:** Which words disappear at `min_df=3`? Are they useful spam clues, or too rare to trust?

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

preview_min_df = 3  # ✏️ Use the same number as MIN_DF_ADVANCED below.
words_to_check = ["free", "prize", "winner", "subscription", "promo", "jackpot", "bitcoin"]  # ✏️ Add your own words.

# In how many TRAINING messages does each word appear? (A repeat inside one message counts once.)
counter = CountVectorizer(binary=True)
word_counts = counter.fit_transform(X_train)
messages_with_word = np.asarray(word_counts.sum(axis=0)).ravel()

def keep_or_remove(count, min_df):
    if count == 0:
        return "⚪ Not in the data"
    return "✅ Keep" if count >= min_df else "❌ Remove"

preview_rows = []
for word in words_to_check:
    position = counter.vocabulary_.get(word.lower())
    count = int(messages_with_word[position]) if position is not None else 0
    preview_rows.append({
        "Word": word,
        "Training messages containing it": count,
        "min_df=1": keep_or_remove(count, 1),
        f"min_df={preview_min_df}": keep_or_remove(count, preview_min_df),
    })
preview_table = pd.DataFrame(preview_rows).sort_values("Training messages containing it", ascending=False)
show_table(preview_table)

vocabulary_total = len(counter.vocabulary_)
removed_total = int((messages_with_word < preview_min_df).sum())
print(f"Whole vocabulary: {vocabulary_total:,} words. Seen in fewer than {preview_min_df} training messages: "
      f"{removed_total:,} ({removed_total / vocabulary_total:.0%}), so min_df={preview_min_df} removes them.")
print("min_df counts messages, not repeats: a word used five times in one message still counts once.")

In [ ]:
# OPTIONAL ADVANCED ONLY. It does not change the selected candidate.
RUN_ADVANCED = False  # ✏️ Set True to do the challenge.
if RUN_ADVANCED:
    MIN_DF_ADVANCED = 3  # ✏️ Experiment setting: 3 means 3 training messages.
    adv = train_and_record("Advanced min_df", alpha=selected["alpha"], min_df=MIN_DF_ADVANCED)
    baseline_vocab = len(selected_pipeline.named_steps["tfidf"].vocabulary_)
    advanced_vocab = len(RUN_MODELS["Advanced min_df"].named_steps["tfidf"].vocabulary_)
    print("Vocabulary sizes | baseline:", baseline_vocab, "| advanced:", advanced_vocab)
    advanced_comparison = pd.DataFrame([
        {"run": "Selected core candidate", "min_df": 1,
         "vocabulary_size": baseline_vocab,
         "val_spam_precision": selected["val_spam_precision"],
         "val_spam_recall": selected["val_spam_recall"]},
        {"run": "Advanced min_df", "min_df": MIN_DF_ADVANCED,
         "vocabulary_size": advanced_vocab,
         "val_spam_precision": adv["val_spam_precision"],
         "val_spam_recall": adv["val_spam_recall"]},
    ])
    display(advanced_comparison)
    advanced_comparison.to_csv(OUT / "advanced_comparison.csv", index=False)
    assert advanced_vocab <= baseline_vocab
else:
    print("Advanced challenge skipped. Core candidate remains unchanged.")

**VERIFY:** With `RUN_ADVANCED=True`, the new vocabulary should be no larger than the original. Validation precision and recall may rise, fall, or stay unchanged.

**EXPLAIN:** Is a smaller vocabulary always better? What evidence supports your answer?

**Hint:** The comparison must use identical train/validation splits and the **same selected alpha**. The optional third run is not part of the two-run core experiment.

### See it — what removing rare words changed (live table + chart)
**WHAT:** Appears only when `RUN_ADVANCED = True`. It sets the selected candidate next to the `min_df` experiment, including the two mistake counts.

**EXPLAIN:** Which numbers got better, which got worse, and which mistake would matter more for automatic blocking?

In [ ]:
if RUN_ADVANCED:
    base_counts = error_counts(selected_pipeline, X_val, y_val)
    adv_counts = error_counts(RUN_MODELS["Advanced min_df"], X_val, y_val)
    # (measure, selected value, advanced value, show as a score (3 decimals) or a count)
    measures = [
        ("Vocabulary size", baseline_vocab, advanced_vocab, "count"),
        ("Spam precision", selected["val_spam_precision"], adv["val_spam_precision"], "score"),
        ("Spam recall", selected["val_spam_recall"], adv["val_spam_recall"], "score"),
        ("False alarms", base_counts["false_alarms"], adv_counts["false_alarms"], "count"),
        ("Missed spam", base_counts["missed_spam"], adv_counts["missed_spam"], "count"),
    ]
    advanced_rows = [{"Measure": "min_df", "Selected candidate": str(selected["min_df"]),
                      "Advanced run": str(MIN_DF_ADVANCED), "Change (advanced - selected)": ""}]
    for measure, old, new, kind in measures:
        if kind == "score":
            old_text, new_text, change_text = f"{old:.3f}", f"{new:.3f}", f"{new - old:+.3f}"
        else:
            old_text, new_text, change_text = f"{old:,}", f"{new:,}", f"{new - old:+,}"
            if measure == "Vocabulary size" and old:
                change_text += f" ({(new - old) / old:+.0%})"
        advanced_rows.append({"Measure": measure, "Selected candidate": old_text,
                              "Advanced run": new_text, "Change (advanced - selected)": change_text})
    show_table(pd.DataFrame(advanced_rows))
    print("Reading the change column: for precision and recall, + is better; for false alarms and missed spam, - is better.")

    names = ["Selected candidate", f"Advanced (min_df={MIN_DF_ADVANCED})"]
    fig, (left, right) = plt.subplots(1, 2, figsize=(10.5, 3.8))
    vocab_bars = left.bar(names, [baseline_vocab, advanced_vocab], color=[GREY, BLUE])
    left.bar_label(vocab_bars, labels=[f"{v:,}" for v in (baseline_vocab, advanced_vocab)])
    left.set_ylim(0, max(baseline_vocab, advanced_vocab) * 1.15)
    left.set_title("Vocabulary size (words kept)")

    positions = np.arange(2)
    width = 0.38
    precision_values = [selected["val_spam_precision"], adv["val_spam_precision"]]
    recall_values = [selected["val_spam_recall"], adv["val_spam_recall"]]
    p_bars = right.bar(positions - width / 2, precision_values, width, color=BLUE, label="Spam precision")
    r_bars = right.bar(positions + width / 2, recall_values, width, color=ORANGE, label="Spam recall")
    right.bar_label(p_bars, fmt="%.3f")
    right.bar_label(r_bars, fmt="%.3f")
    right.set_xticks(positions, names)
    right.set_ylim(0, 1.15)
    right.set_title("Validation scores (higher is better)")
    right.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
    fig.tight_layout()
    plt.show()
    print()

    show_glossary([
        ("min_df", "Minimum number of training messages a word must appear in to be kept.",
         "Smaller vocabulary: rare words (typos, numbers, names) are dropped, and some useful spam clues may go with them.",
         "Larger vocabulary: every word is kept, including one-off words that may only be noise."),
        ("Vocabulary size", "Number of distinct words the TF-IDF stage keeps.",
         "More detail, but a bigger chance of memorizing rare words.",
         "A simpler model, but rare and telling words may be lost."),
    ])
else:
    print("Advanced challenge skipped. Set RUN_ADVANCED = True in the cell above, rerun it, then rerun this cell.")

### Interpreting the `min_df` comparison

*Reference results (seed 42, the selected candidate is Run A with `alpha` 0.5, compared with `min_df` = 3).*

| Measure | `min_df` = 1 (selected) | `min_df` = 3 (advanced) | Change |
|---|---:|---:|---:|
| Vocabulary size | 6,524 | 1,813 | −4,711 (−72%) |
| Spam precision | 1.000 | 0.991 | −0.009 |
| Spam recall | 0.750 | 0.852 | +0.102 |
| False alarms | 0 | 1 | +1 |
| Missed spam | 32 | 19 | −13 |

**What this means:** Dropping words that appear in fewer than 3 training messages cut the vocabulary by about 72%. In this reference run recall *rose*, so a smaller vocabulary was not automatically worse. The price was one false alarm: a legitimate message was blocked. For automatic blocking, that is the mistake that matters most, so the precision-first rule from Step 6 would keep `min_df` = 1. For a review folder, `min_df` = 3 looks attractive. One validation run is thin evidence, though: a single message moves precision by about 0.01.

## Step 9 — Final verification and export
**WHY:** Session 4 needs the saved pipeline plus information about how it was selected.

**WHAT:** Optionally evaluate the **already selected** model on the held-out test set once. Export the model and its traceability files together.

**Predict:** Can a model chosen on validation perform differently on a fresh test set?

In [ ]:
# Optional ONE-TIME evaluation only after candidate selection.
RUN_FINAL_TEST = False  # Instructor may allow setting True after selection is frozen.
if RUN_FINAL_TEST:
    test_predictions = reloaded_pipeline.predict(X_test)
    test_summary = {
        "selected_run": candidate_name,
        "test_spam_precision": float(precision_score(y_test, test_predictions,
                                                    pos_label="spam", zero_division=0)),
        "test_spam_recall": float(recall_score(y_test, test_predictions,
                                                  pos_label="spam", zero_division=0)),
        "note": "Do not use these scores to change the candidate or tune alpha.",
    }
    (OUT / "test_evaluation.json").write_text(json.dumps(test_summary, indent=2), encoding="utf-8")
    print("Final test evaluation:", test_summary)
else:
    print("Test set untouched: core requirement satisfied.")

import zipfile
handoff_readme = OUT / "README_SESSION4.txt"
handoff_readme.write_text(
    "Session 3 -> Session 4: Load selected_pipeline.joblib ONLY from a trusted source.\n"
    "It is a fitted scikit-learn Pipeline: TfidfVectorizer + MultinomialNB.\n"
    "Input to predict(): a list of raw SMS strings.\n"
    "selection_note.json describes the chosen validation candidate.\n"
    "This is NOT a registered MLflow model version.\n",
    encoding="utf-8"
)
required = [OUT / "selected_pipeline.joblib", OUT / "run_results.csv",
            OUT / "selection_note.json", handoff_readme]
optional = [OUT / "test_evaluation.json", OUT / "advanced_comparison.csv"]
keep = required + [p for p in optional if p.exists()]
assert all(p.exists() and p.stat().st_size for p in required)
assert reloaded_pipeline.predict(["Hello, see you soon!"]).shape == (1,)
zip_path = Path("session3_sms_handoff.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for p in keep:
        bundle.write(p, arcname=p.name)
with zipfile.ZipFile(zip_path) as bundle:
    assert set(p.name for p in required).issubset(set(bundle.namelist()))
print("SUCCESS! Handoff bundle:", zip_path.resolve())
print("Included:", [p.name for p in keep])

# In Colab, the next line offers a direct file download.
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Local Jupyter: download the ZIP from the notebook's working folder.")

**VERIFY:** `session3_sms_handoff.zip` contains `selected_pipeline.joblib`, `run_results.csv`, `selection_note.json`, and `README_SESSION4.txt`.

**EXPLAIN / exit ticket (write short answers):**
1. Which single setting differed between Run A and Run B?
2. Why did you choose your candidate? Refer to **validation** precision and recall.
3. How did you confirm that the **whole pipeline** was saved and can be reused?
4. What does an MLflow run ID tell you? What is the fallback if tracking is unavailable?
5. Why didn't we select a model using the test set?

**Submit:** The handoff ZIP, your selection reason, and one observation from Exercise 4. If you completed the advanced challenge, add your vocabulary-size and metric comparison.

**Next session:** Use the selected fitted pipeline in an automated, repeatable training workflow.

Official references: [UCI dataset](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) · [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking/) · [MLflow sklearn API](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.sklearn.html) · [scikit-learn leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html)

### See it — what is inside the handoff bundle (live table)
**WHY:** Session 4 only gets the ZIP. Every file in it should have a clear job.

**VERIFY:** The first table is read from the ZIP itself, not from your working folder. **EXPLAIN:** For each exit-ticket question, which file holds the evidence?

In [ ]:
bundle_notes = {
    "selected_pipeline.joblib": ("The fitted TF-IDF + Naive Bayes pipeline",
                                 "Load it (trusted source only) and call predict() on raw SMS strings"),
    "run_results.csv": ("Both core runs: settings, validation scores, MLflow IDs",
                        "See what was compared when the candidate was chosen"),
    "selection_note.json": ("Chosen run, reason, seed, versions, and whether test data was used",
                            "Trace and reproduce the selection"),
    "README_SESSION4.txt": ("Plain-language handoff and safety notes",
                            "Tell the next person how to load the model safely"),
    "test_evaluation.json": ("One-time test scores (only if RUN_FINAL_TEST = True)",
                             "Report the final check without re-tuning"),
    "advanced_comparison.csv": ("The min_df experiment (only if RUN_ADVANCED = True)",
                                "Show the optional experiment"),
}
with zipfile.ZipFile(zip_path) as bundle:
    bundle_rows = []
    for item in bundle.infolist():
        what, use = bundle_notes.get(item.filename, ("(not described)", ""))
        bundle_rows.append({"File in the ZIP": item.filename, "Size (KB)": item.file_size / 1024,
                            "What it holds": what, "Session 4 uses it to": use})
show_table(pd.DataFrame(bundle_rows), {"Size (KB)": "{:,.1f}"})

display(Markdown("**Your handoff note at a glance** (read back from `selection_note.json`)"))
saved_note = json.loads((OUT / "selection_note.json").read_text(encoding="utf-8"))
run_id = saved_note["mlflow_run_id"]
show_table(pd.DataFrame([
    ("Which run did you select?", "selected_run", str(saved_note["selected_run"])),
    ("Which setting differed between the runs?", "alpha (selected value)", str(saved_note["alpha"])),
    ("How good was it on validation?", "validation_spam_precision / _recall",
     f"{saved_note['validation_spam_precision']:.3f} / {saved_note['validation_spam_recall']:.3f}"),
    ("Why did you choose it?", "selection_reason", saved_note["selection_reason"]),
    ("What does the MLflow run ID tell you?", "mlflow_run_id",
     run_id if run_id == "not_logged" else run_id[:8] + "..."),
    ("Was the test set used to choose?", "test_used_for_selection", str(saved_note["test_used_for_selection"])),
], columns=["Exit-ticket question", "Field in selection_note.json", "Value in your handoff"]))

---
## Step 10 — OPTIONAL BONUS: Does "better" depend on the goal? (Run C)
**WHY:** In Step 6 both goals picked the same run, so the question *"Would a review folder change your pick?"* had an easy answer. Here we build a model that makes it a real question.

**WHAT:** Train **Run C** exactly like Run A (same data, same `alpha`) with **one** change: `fit_prior=False`. Naive Bayes then stops starting from the belief that *most messages are ham*. Next, rank Runs A, B and C under both goals again.

**Predict:** Will Run C find more spam than Run A? What might that cost?

**Rules for this bonus:** Validation data only. The test set stays untouched, and your selected candidate and the Step 9 handoff ZIP do **not** change.

In [ ]:
# BONUS - Run C is Run A with ONE change: fit_prior=False.
ALPHA_C = ALPHA_A  # ✏️ YOUR TURN (after the first run): try ALPHA_B or 2.0, rerun, and watch precision and recall.
run_c = train_and_record("Run C", alpha=ALPHA_C, fit_prior=False)

bonus_names = ["Run A", "Run B", "Run C"]
bonus_rows = []
for run_name in bonus_names:
    run_record = RUN_RECORDS[run_name]
    counts = error_counts(RUN_MODELS[run_name], X_val, y_val)
    bonus_rows.append({
        "Run": run_name,
        "alpha": run_record["alpha"],
        "fit_prior": run_record["fit_prior"],
        "Spam precision": run_record["val_spam_precision"],
        "Spam recall": run_record["val_spam_recall"],
        "False alarms": counts["false_alarms"],
        "Missed spam": counts["missed_spam"],
    })
bonus_table = pd.DataFrame(bonus_rows)
show_table(bonus_table, {"alpha": "{:g}", "Spam precision": "{:.3f}", "Spam recall": "{:.3f}"})

# Trade-off map: every run is one point. Up = fewer real messages blocked; right = less spam gets through.
point_colours = {"Run A": BLUE, "Run B": GREY, "Run C": ORANGE}
fig, ax = plt.subplots(figsize=(6.8, 4.4))
for _, row in bonus_table.iterrows():
    ax.scatter(row["Spam recall"], row["Spam precision"], s=130,
               color=point_colours[row["Run"]], zorder=3)
    ax.annotate(f'{row["Run"]}\n(recall {row["Spam recall"]:.3f},\nprecision {row["Spam precision"]:.3f})',
                (row["Spam recall"], row["Spam precision"]),
                textcoords="offset points", xytext=(8, -40), fontsize=9)
ax.set_xlim(max(0, bonus_table["Spam recall"].min() - 0.15), min(1.05, bonus_table["Spam recall"].max() + 0.1))
ax.set_ylim(max(0, bonus_table["Spam precision"].min() - 0.1), 1.03)
ax.set_xlabel("Spam recall (higher = less spam gets through)")
ax.set_ylabel("Spam precision (higher = fewer real messages blocked)")
ax.set_title("Trade-off map on validation data")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
print("A run that is higher AND further right than another is better on both measures. Runs that are higher but further left (or the reverse) are a trade-off.\n")

# What fit_prior changes (the training counts are live).
ham_train, spam_train = int((y_train == "ham").sum()), int((y_train == "spam").sum())
display(Markdown("**What `fit_prior` changes**"))
show_table(pd.DataFrame([
    ("Starting belief before reading any word",
     f"Learned from training: {ham_train:,} ham vs {spam_train:,} spam (about {ham_train / spam_train:.0f} to 1 for ham)",
     "50 / 50: spam is not assumed to be rare"),
    ("What the words must do", "Overcome the head start that ham has", "Decide on their own"),
    ("Tends to give", "Higher precision, lower recall", "Lower precision, higher recall"),
    ("Fits best when", "A false alarm is costly (automatic blocking)", "Missed spam is costly (review folder)"),
], columns=["Aspect", "fit_prior=True (default; Runs A and B)", "fit_prior=False (Run C)"]))
print("\nSame lever as the threshold in Session 2's optional step: a different way to make the model stricter or more eager.")

### The review-folder question, again — now with a real choice
**WHY:** Step 6 ranked two runs. Now the same two rules rank three, and the goals can disagree.

**RUN:** Rank Runs A, B and C by *automatic blocking* (precision first) and by *review folder* (recall first), then see what each winner costs in real messages.

**VERIFY:** The table names a winner for each goal, and the sentences below it count the same messages as the table above. **EXPLAIN:** Would a review folder change your pick now? Why, or why not?

In [ ]:
bonus_results = pd.DataFrame([RUN_RECORDS[name] for name in bonus_names])
goal_rules = [
    ("Automatic blocking", "val_spam_precision", "val_spam_recall", "precision, then recall"),
    ("Review folder", "val_spam_recall", "val_spam_precision", "recall, then precision"),
]
goal_rows = []
for goal, first_metric, second_metric, rule_text in goal_rules:
    ranked = bonus_results.sort_values([first_metric, second_metric], ascending=[False, False], kind="stable")
    winner = ranked.iloc[0]
    counts = error_counts(RUN_MODELS[winner["run"]], X_val, y_val)
    goal_rows.append({
        "Goal": goal,
        "Ranking rule": rule_text,
        "Winner": winner["run"],
        "Spam precision": winner["val_spam_precision"],
        "Spam recall": winner["val_spam_recall"],
        "Real messages wrongly flagged": counts["false_alarms"],
        "Spam still delivered": counts["missed_spam"],
    })
goal_table = pd.DataFrame(goal_rows)
show_table(goal_table, {"Spam precision": "{:.3f}", "Spam recall": "{:.3f}"})

blocking, review = goal_rows
if blocking["Winner"] == review["Winner"]:
    print("Both goals pick", blocking["Winner"], "- the goal does not change the winner here.")
else:
    print("The goal changes the winner.\n")
    print(f"Automatic blocking -> {blocking['Winner']}: {blocking['Real messages wrongly flagged']} real messages "
          f"are blocked with no one checking; {blocking['Spam still delivered']} spam messages are delivered.")
    print(f"Review folder      -> {review['Winner']}: {review['Real messages wrongly flagged']} real messages "
          f"land in the folder, where a person can rescue them; {review['Spam still delivered']} spam messages are delivered.")

### How to answer the review-folder question

Ask two things: **What happens to a flagged SMS?** and **Which mistake hurts more?**

| What happens to a flagged SMS | Cost of a false alarm | Cost of missed spam | Better fit |
|---|---|---|---|
| It is deleted or hidden automatically | **High:** a real message (a login code, a reminder) is lost for good | Medium: spam reaches the inbox | Precision first |
| It is moved to a folder a person checks | **Low:** the real message is only delayed | **High:** spam is delivered | Recall first |

### Interpreting the three runs

*Reference results (seed 42, validation: 1,032 messages, 128 of them spam). If you kept the defaults you should see the same numbers.*

| Measure | Run A (`alpha` 0.5) | Run B (`alpha` 1.0) | Run C (`alpha` 0.5, `fit_prior=False`) |
|---|---:|---:|---:|
| Spam precision | 1.000 | 1.000 | 0.902 |
| Spam recall | 0.750 | 0.633 | 0.938 |
| False alarms | 0 | 0 | 13 |
| Missed spam | 32 | 47 | 8 |

| Goal | Winner | What it costs |
|---|---|---|
| Automatic blocking (precision, then recall) | Run A | 0 real messages blocked, 32 spam delivered |
| Review folder (recall, then precision) | Run C | 13 real messages held in the folder, 8 spam delivered |

**What this means:** Compared with Run A, Run C catches **24 more spam messages** (96 → 120) but also flags **13 real messages** that Run A left alone. For automatic blocking, 13 vanished real messages is a high price, so Run A stays the better pick. For a review folder, those 13 only need a glance and 24 fewer spam messages get through, so Run C is the better pick. **Yes, the review folder changes the pick, and the reason is the cost of each mistake, not the data.** Neither answer is wrong; the application decides.

**Remember:** Your selected candidate and the Step 9 handoff are still based on Steps 6–9. To hand off Run C, repeat Steps 6–9 with it and say why a review folder is the goal. With only 128 spam messages in validation, the exact numbers shift if you change the random seed.

**EXPLAIN / exit ticket (bonus):**
1. Which single setting differs between Run A and Run C?
2. Did Run C find more spam? What did it cost?
3. Would a review folder change your pick? Use the false-alarm and missed-spam counts in your answer.
4. Why did we not use the test set to choose between Runs A and C?